In [0]:
# Databricks notebook source

from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col, broadcast, collect_set, size, array_contains


class Transformer:

    def __init__(self):
        pass

    def transform(self, inputDFs):
        raise NotImplementedError(
            "Subclasses must implement transform()"
        )


class AirpodsAfterIphoneTransformer(Transformer):

    def transform(self, inputDFs):
        """Customers who bought AirPods after buying iPhone"""

        transcatioInputDF = inputDFs.get("transcatioInputDF")

        windowSpec = Window.partitionBy("customer_id").orderBy("transaction_date")

        transformedDF = transcatioInputDF.withColumn(
            "next_product_name", lead("product_name").over(windowSpec)
        )

        filteredDF = transformedDF.filter(
            (col("product_name") == "iPhone") &
            (col("next_product_name") == "AirPods")
        )

        customerInputDF = inputDFs.get("customerInputDF")

        joinDF = customerInputDF.join(
            broadcast(filteredDF),
            "customer_id"
        )

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )


class OnlyAirpodsAndIphone(Transformer):

    def transform(self, inputDFs):
        """Customers who bought ONLY iPhone and AirPods, nothing else"""

        transcatioInputDF = inputDFs.get("transcatioInputDF")

        groupedDF = transcatioInputDF.groupBy("customer_id").agg(
            collect_set("product_name").alias("products")
        )

        filteredDF = groupedDF.filter(
            (array_contains(col("products"), "iPhone")) &
            (array_contains(col("products"), "AirPods")) &
            (size(col("products")) == 2)
        )

        customerInputDF = inputDFs.get("customerInputDF")

        joinDF = customerInputDF.join(
            broadcast(filteredDF),
            "customer_id"
        )

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )
class IphoneWithoutAirpods(Transformer):

    def transform(self, inputDFs):
        """Customers who bought iPhone but NEVER bought AirPods"""

        transcatioInputDF = inputDFs.get("transcatioInputDF")

        # Group by customer and collect all products they bought
        groupedDF = transcatioInputDF.groupBy("customer_id").agg(
            collect_set("product_name").alias("products")
        )

        print("All customers with their products:")
        display(groupedDF)

        # Keep only customers who have iPhone in list BUT no AirPods
        filteredDF = groupedDF.filter(
            (array_contains(col("products"), "iPhone")) &
            (~array_contains(col("products"), "AirPods"))
        )

        print("Customers with iPhone but no AirPods:")
        display(filteredDF)

        customerInputDF = inputDFs.get("customerInputDF")

        joinDF = customerInputDF.join(
            broadcast(filteredDF),
            "customer_id"
        )

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )